### Recover Complaint Factor 분석 

In [5]:
"""
분기별 Recovery / 물리결함 PPM 집계 스크립트
- 입력 1: Recovery 리뷰 데이터 (dt, asin, cf1, collection, title, review ...)
- 입력 2: 월별 출하 데이터 (yr_month, asin, collection, shipped_units ...)
- 출력: 분기별 출하대수 / Recovery 건수·PPM / 물리결함 건수·PPM

PPM = (결함 리뷰 수 ÷ 출하대수) × 1,000,000
물리결함 = Height(두께/팽창 미달) 또는 Corners/edges(모서리·측면 미팽창·변형)
"""

import re
import pandas as pd

REVIEW_CSV = "data_202606171248.csv"   # Recovery 리뷰
SALES_CSV  = "data_202606171525.csv"   # 월별 출하 (box_type 있는 버전)


# ---------------------------------------------------------------------------
# 1) 물리결함 분류기 (리뷰 텍스트 기반)
# ---------------------------------------------------------------------------
def is_height(t: str) -> bool:
    """광고 대비 두께/높이 미달 또는 미팽창."""
    patterns = [
        r'not\s+\d{1,2}\s*("|in|inch)',
        r'(only|barely|about|just)\s+\d{1,2}\s*("|in|inch)',
        r'\d{1,2}\s*("|inch|in)\b.{0,30}(only|instead|not|short|thin)',
        r'(thinner|less thick|not.{0,10}thick|much thinner)\b',
        r'thin\b.{0,30}(advertis|expect|describ|than|disappoint|son|comfort|cheap)',
        r'(never|not|did(n.t| not)|won.t|hasn.t|fail).{0,30}'
        r'(expand|inflat|fluff|rise|decompress|reach|resize|full size|full height)',
        r'(expand|inflat|fluff|rise|decompress).{0,20}(only|never|not|fail|partial|halfway)',
        r'(thickness|height).{0,30}(not|less|short|advertis|claim|disappoint|wrong)',
    ]
    return any(re.search(p, t) for p in patterns)


def is_corner(t: str) -> bool:
    """모서리/측면/끝부분 미팽창 또는 변형."""
    if re.search(r'\b(corner|edge|edges|sides?|ends?)\b', t):
        if re.search(r'(corner|edge|side|end).{0,40}'
                     r'(not|never|round|slant|uneven|out of shape|won.t|didn.t|'
                     r'flat|lump|square|expand|rise|fold|crook)', t):
            return True
        if re.search(r'(not|never|round|slant|uneven|won.t|didn.t|square|expand|'
                     r'rise|fold).{0,40}(corner|edge|side|end)', t):
            return True
    return False


# ---------------------------------------------------------------------------
# 2) 데이터 적재
# ---------------------------------------------------------------------------
def load_reviews(path: str) -> pd.DataFrame:
    r = pd.read_csv(path)
    r["dt"] = pd.to_datetime(r["dt"])
    r["q"] = r["dt"].dt.to_period("Q")
    text = (r["title"].fillna("") + " || " + r["review"].fillna("")).str.lower()
    r["is_phys"] = text.map(lambda t: is_height(t) or is_corner(t))
    return r


def load_sales(path: str) -> pd.DataFrame:
    s = pd.read_csv(path, dtype={"yr_month": str})
    s["shipped_units"] = pd.to_numeric(s["shipped_units"], errors="coerce").fillna(0)
    s["q"] = pd.to_datetime(s["yr_month"], format="%Y%m").dt.to_period("Q")
    return s


# ---------------------------------------------------------------------------
# 3) 분기별 PPM 집계
# ---------------------------------------------------------------------------
def quarterly_ppm(reviews: pd.DataFrame, sales: pd.DataFrame) -> pd.DataFrame:
    units = sales.groupby("q")["shipped_units"].sum()
    recovery_cnt = reviews.groupby("q").size()
    phys_cnt = reviews.groupby("q")["is_phys"].sum()

    df = pd.DataFrame({
        "shipped_units": units,
        "recovery_cnt": recovery_cnt,
        "phys_cnt": phys_cnt,
    }).fillna(0)

    # 비율 집계 원칙: 분자/분모를 각각 합산한 뒤 나눔 (PPM 평균을 다시 평균내지 않음)
    df["recovery_ppm"] = (df["recovery_cnt"] / df["shipped_units"] * 1e6).round(1)
    df["phys_ppm"] = (df["phys_cnt"] / df["shipped_units"] * 1e6).round(1)

    for c in ["shipped_units", "recovery_cnt", "phys_cnt"]:
        df[c] = df[c].astype(int)

    df = df.reset_index()
    df["q"] = df["q"].astype(str)
    return df[["q", "shipped_units", "recovery_cnt", "recovery_ppm",
               "phys_cnt", "phys_ppm"]]


# ---------------------------------------------------------------------------
# 4) 실행
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    reviews = load_reviews(REVIEW_CSV)
    sales = load_sales(SALES_CSV)
    result = quarterly_ppm(reviews, sales)

    result.columns = ["분기", "출하대수", "Recovery 건수",
                      "Recovery PPM", "물리결함 건수", "물리결함 PPM"]
    print(result.to_string(index=False))
    # 참고: 가장 최근 분기는 부분 데이터일 수 있으므로 해석 시 주의(예: 2026Q2 = 6/10까지)

    분기   출하대수  Recovery 건수  Recovery PPM  물리결함 건수  물리결함 PPM
2024Q1 294918           97         328.9       57     193.3
2024Q2 288566           68         235.6       41     142.1
2024Q3 341617           67         196.1       38     111.2
2024Q4 295848           85         287.3       48     162.2
2025Q1 386978          127         328.2       76     196.4
2025Q2 330047           71         215.1       38     115.1
2025Q3 484523           87         179.6       40      82.6
2025Q4 274762           62         225.6       36     131.0
2026Q1 285818          107         374.4       56     195.9
2026Q2 272005           67         246.3       48     176.5
